In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace ,HuggingFaceEndpoint
import os
import langchain

In [2]:
load_dotenv()

python-dotenv could not parse statement starting at line 1


True

In [3]:

hf_token = os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")

# 1. Initialize the HuggingFaceEndpoint LLM
# Using Qwen/Qwen2.5-72B-Instruct (hosted on free HF Inference API)
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=hf_token,
    max_new_tokens=150,
    temperature=0.7,
)

# 2. Wrap it with ChatHuggingFace
model = ChatHuggingFace(llm=llm)



In [4]:
class BlogState(TypedDict):
    title:str
    outline : str
    content: str
    

In [5]:
def create_outline(state:BlogState) ->BlogState:
    title= state['title']
    prompt = f'Generate a detailed outline for a  blog of topic{title}'
    outline=llm.invoke(prompt).content
    state['outline']=outline
    return state


In [11]:
def create_blog(state:BlogState) ->BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'write a detailed blog on the title-{title} using the following outline \n {outline}'

    content = llm.invoke(prompt).content
    state['content']=content
    return state 


In [ ]:
def create_evaluate(state :BlogState) ->BlogState:
    title=state['title']
    outline=state['outline']
    blog=state['blog']

    prompt=f'rate of blog {blog} based on theseout;ine{outline}
    
    


In [ ]:
graph = StateGraph(BlogState)
#nodes

graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('create_evaluate',create_evaluate)

#edges
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

workflow=graph.compile()



In [15]:
final_result= workflow.invoke({'title' : 'Rise of AI in India'})
print(final_result)

ValueError: Model Qwen/Qwen2.5-72B-Instruct is not supported for task text-generation and provider novita. Supported task: conversational.